# Step 6.0 — Sanity Check of the Final Model (XGBoost tuned, 6 months)

## Objective
Validate the operating point (threshold) of the final XGBoost model:
- the model returns probabilities `P(rapid)`
- the `rapid/slow` decision depends on the threshold

In this sanity check:
- we compare reference thresholds
- we sweep a grid to observe the trade-off (precision, recall, F2, FP/FN)
- we choose a final operational threshold
- we save tables/figures as evidence of the decision

## Inputs
- `01_data/processed/dataset_6m_v1.csv`
- `models/final_xgb_6m.joblib`
- `models/final_xgb_6m_metadata.json`

## Outputs
- `04_outputs/tables/step6_sanity_xgb_threshold_checks.csv`
- `04_outputs/tables/step6_sanity_xgb_threshold_curve.csv`
- (optional) `04_outputs/figures/step6_sanity_xgb_f2_vs_threshold.png`


In [ ]:
import os, json
import numpy as np
import pandas as pd
import joblib

from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score, fbeta_score

DATASET_6M = os.path.join("..", "01_data", "processed", "dataset_6m_v1.csv")
MODEL_PATH = os.path.join("..", "models", "final_xgb_6m.joblib")
META_PATH  = os.path.join("..", "models", "final_xgb_6m_metadata.json")

df = pd.read_csv(DATASET_6M)
pipe = joblib.load(MODEL_PATH)

with open(META_PATH, "r", encoding="utf-8") as f:
    meta = json.load(f)

slope_col = meta["slope_col"]
slope_cutoff = meta["slope_cutoff_30pct"]
feat_cols = meta["feature_cols"]

y_true = (df[slope_col] <= slope_cutoff).astype(int).to_numpy()
X = df[feat_cols].copy()

proba = pipe.predict_proba(X)[:, 1]

print("N:", len(df), "| rapid% true:", y_true.mean())
print("slope_cutoff:", slope_cutoff)
print("proba min/mean/max:", float(proba.min()), float(proba.mean()), float(proba.max()))

In [ ]:
def eval_at_threshold(y_true, proba, thr, beta=2):
    y_pred = (proba >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0,1]).ravel()

    return {
        "thr": float(thr),
        "pred_pos_rate": float(y_pred.mean()),
        "TN": int(tn), "FP": int(fp), "FN": int(fn), "TP": int(tp),
        "precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "recall": float(recall_score(y_true, y_pred, zero_division=0)),
        "F1": float(f1_score(y_true, y_pred, zero_division=0)),
        "F2": float(fbeta_score(y_true, y_pred, beta=beta, zero_division=0)),
    }

# 3 reference thresholds:
# - 0.50 (neutral)
# - thr_30pct_pred (to produce ~30% predicted positives)
# - thr_from_cv (mean threshold chosen per fold in Step 5 for XGB tuned)
thr_30pct_pred = float(np.quantile(proba, 1 - 0.30))
thr_from_cv = 0.19  # from step5_xgb_tuned_results_6m.csv (thr_decision_mean ≈ 0.19)

checks = [
    eval_at_threshold(y_true, proba, thr_from_cv),
    eval_at_threshold(y_true, proba, 0.50),
    eval_at_threshold(y_true, proba, thr_30pct_pred),
]
checks_df = pd.DataFrame(checks)
checks_df


In [ ]:
ths = np.linspace(0.05, 0.95, 19)
rows = [eval_at_threshold(y_true, proba, t) for t in ths]
curve = pd.DataFrame(rows)[["thr","pred_pos_rate","precision","recall","F2","FP","FN"]]
curve

In [ ]:
OUT_TABLES = os.path.join("..", "04_outputs", "tables")
OUT_FIGS   = os.path.join("..", "04_outputs", "figures")
os.makedirs(OUT_TABLES, exist_ok=True)
os.makedirs(OUT_FIGS, exist_ok=True)

checks_path = os.path.join(OUT_TABLES, "step6_sanity_xgb_threshold_checks.csv")
curve_path  = os.path.join(OUT_TABLES, "step6_sanity_xgb_threshold_curve.csv")

checks_df.to_csv(checks_path, index=False)
curve.to_csv(curve_path, index=False)

print("Saved:", checks_path)
print("Saved:", curve_path)


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8,4))
plt.plot(curve["thr"], curve["F2"], marker="o")
plt.xlabel("threshold")
plt.ylabel("F2")
plt.title("Sanity check — XGBoost tuned (6m): F2 vs threshold")
plt.grid(True, alpha=0.3)
plt.tight_layout()

fig_path = os.path.join(OUT_FIGS, "step6_sanity_xgb_f2_vs_threshold.png")
plt.savefig(fig_path, dpi=300)
plt.show()
print("Saved:", fig_path)


In [ ]:
import json

META_PATH  = os.path.join("..", "models", "final_xgb_6m_metadata.json")

FINAL_THR = 0.25  

with open(META_PATH, "r", encoding="utf-8") as f:
    meta = json.load(f)

meta["decision_threshold_old"] = meta.get("decision_threshold", None)
meta["decision_threshold"] = float(FINAL_THR)
meta["decision_threshold_reason"] = "sanity-check trade-off: chosen to balance recall and operational load"

with open(META_PATH, "w", encoding="utf-8") as f:
    json.dump(meta, f, ensure_ascii=False, indent=2)

print("Updated:", META_PATH)
print("old -> new:", meta["decision_threshold_old"], "->", meta["decision_threshold"])
